# 06 — The Capstone on Real Hardware, and the Frontier

In Module 04 you asked whether a quantum coupling measure reads something the phase-locking value (PLV) cannot, and you wrote down what the measurements showed. With an embedding that carries the second harmonic it reads past PLV. A classical statistic of the same two harmonics reads as far, from shorter records. What quantum optimal transport brings is one common geometry. Every number in that accounting was computed on a classical computer, from density matrices written out exactly. Here we ask the hardware question that closes the course. The two ensembles of `04/15` have different quantum mutual informations (QMI), and we call that difference a **gap**. It is **0.0044 nat** with the equal-weight embedding of `04/15`, and **0.0145 nat** once the first harmonic is turned down to a weight of 0.2, as in `04/17`. Is a real quantum processor sharp enough to resolve gaps that small? We will answer with numbers, and the answer will show where the frontier lies.

**Prerequisites:** `04/15_discriminating_experiment`, `04/16_capstone_synthesis`, `04/17_embedding_design_amplifies_the_effect`, `04/18_what_no_classical_data_embedding_can_buy`, `04/19_vqe_limitation_de_palma`, `04/20_frontier_synthesis`, `05/04_quantum_mutual_information_on_real_hardware`.

**What you'll be able to do**
- Measure `I(A:B)` on the device with a genuine statistical error bar (bootstrap-resampled from the counts).
- Quantify the device's *systematic* bias — the gap between the ideal QMI and what the chip returns.
- Compare both of those errors, side by side, to the two gaps of the capstone.
- Estimate the number of shots a device would need to resolve each gap, and say what more shots cannot fix.
- State the bottom line of the course: what the theory established, and what a machine charges to read it.

You have measured `I(A:B)` on a chip already, in `05/04`, and watched decoherence steal a piece of it. Now you turn that same measurement into an instrument and point it at the capstone: you ask how finely a real device can resolve a mutual-information difference, and you compare that resolution to the gaps you computed. This is the most candid notebook in the course — and candour, here, is the science.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.runtime import select_backend
from qot_course.hardware.tomography import two_qubit_tomography_circuits, density_from_counts
from qot_course.infotheory.quantum import quantum_mutual_information
from qot_course.quantum.composite import bell_state
from qot_course.quantum.density import density_matrix

np.random.seed(42)
viz.use_course_style()

# Gaps in quantum mutual information between the two matched-PLV ensembles of 04/15, both
# computed there on 150,000 phase samples per ensemble and printed in section 1 of 04/17.
EQUAL_WEIGHT_GAP_NATS = 0.0044  # qutrit embedding with equal weights (04/15)
WEIGHTED_GAP_NATS = 0.0145      # first harmonic turned down to weight 0.2 (04/17)
CAPSTONE_GAPS = {"equal weights": EQUAL_WEIGHT_GAP_NATS, "weighted": WEIGHTED_GAP_NATS}

## 1 · What we are testing

Let us be precise about what was settled before this notebook, and what was not. In `04/15` you built two ensembles of phase differences with the same PLV and a different second moment. On 150,000 samples each, the equal-weight qutrit embedding gives them quantum mutual informations that differ by **0.0044 nat**. With the first harmonic turned down to a weight of 0.2, `04/17` brought the gap to **0.0145 nat**. Both numbers are computed on a **classical computer**: each density matrix is built from the samples and written out in full, with no measurement and no read-out noise. Module 04 also settled what the gaps mean. The second-harmonic PLV reads the same difference, from fewer samples than the equal-weight measure needs and level with the weighted one. No quantum device is needed to compute any of it.

This notebook asks a **different** question: **what is a real processor's resolution for `I(A:B)`?** How finely can a physical device, with its finite shots and its gate and read-out errors, pin down a mutual-information value at all? Two sources of uncertainty must stay apart, because conflating them would blur the lesson:

- the **sampling of the signals**: how many phase samples a statistic needs to tell the two ensembles apart. That was the subject of `04/15` to `04/17`, and it is a statement about the data;
- the **σ_stat** we are about to measure, a *device error bar*: the statistical scatter of a single hardware estimate of a mutual information, for a state that is known exactly. That is a statement about the chip.

A demonstration on hardware would pay both costs, and preparing the states on a device, sample by sample, could tie the two together. Here we measure the second one alone, on the most favourable footing: the state is given exactly, and only the read-out of the device stands between us and its mutual information. Preparing the capstone's states on a device is not tested in this notebook.

## 2 · Measuring the device's QMI resolution

To measure the device's resolution we use the cleanest probe we have: a Bell pair, whose ideal mutual information we know exactly. We reconstruct its joint state by two-qubit tomography — the nine Pauli settings of `05/04` — and read off `I(A:B)`. Each setting is measured 8,192 times: 8,192 *shots per setting*. We work in **nats** throughout (natural log), so the device's numbers are directly comparable to the capstone's gaps, which are in nats.

A single QMI estimate is not enough; we need to know how much it would *wobble*. So we characterise two separate errors:

- a **statistical error bar** `σ_stat`, obtained by bootstrap-resampling each setting's counts (drawing new multinomial counts with the same totals) and recomputing QMI many times — the scatter of those re-estimates is the statistical uncertainty, and it costs no new jobs on the device;
- a **systematic bias**, the gap `I_ideal − I_device`: the device's QMI sits *below* the ideal because decoherence and read-out error leak correlation, and that offset does not average away.

The first shrinks with more shots; the second does not. Holding them apart is the whole point. One more word before the code: a gap between two ensembles is a difference of two estimates, so we will read the gaps against the **band of a difference**, `√2·σ_stat`.

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")
if not is_real:
    backend.set_options(seed_simulator=7)  # reproducible offline figure; a real QPU has no such option

prep = QuantumCircuit(2)
prep.h(0)
prep.cx(0, 1)                               # a Bell pair: our QMI probe
circuits = two_qubit_tomography_circuits(prep)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
settings = list(circuits)
res = SamplerV2(mode=backend).run([pm.run(circuits[s]) for s in settings], shots=8192).result()
counts_by_setting = {s: res[i].data.c.get_counts() for i, s in enumerate(settings)}

qmi_ideal = quantum_mutual_information(density_matrix(bell_state()), dims=[2, 2], base=np.e)   # nats
qmi_point = quantum_mutual_information(density_from_counts(counts_by_setting, n_qubits=2), dims=[2, 2], base=np.e)

# Statistical error bar: bootstrap-resample each setting's counts (multinomial) -- cheap, no new jobs.
rng = np.random.default_rng(0)
boot = []
for _ in range(200):
    resampled = {}
    for s, counts in counts_by_setting.items():
        ks = list(counts)
        ps = np.array([counts[k] for k in ks], dtype=float)
        n = int(ps.sum())
        resampled[s] = {k: int(v) for k, v in zip(ks, rng.multinomial(n, ps / ps.sum()))}
    boot.append(quantum_mutual_information(density_from_counts(resampled, n_qubits=2), dims=[2, 2], base=np.e))
sigma_stat = float(np.std(boot))
bias = float(qmi_ideal - qmi_point)
print(f"I(A:B) ideal  = {qmi_ideal:.4f} nats")
print(f"I(A:B) device = {qmi_point:.4f} +/- {sigma_stat:.4f} (stat) nats")
print(f"systematic bias (ideal - device) = {bias:.4f} nats")
band = float(np.sqrt(2) * sigma_stat)  # statistical band of a DIFFERENCE of two estimates
print(f"band of a difference = sqrt(2) * sigma_stat = {band:.4f} nats  (in a difference the bias cancels only in part: see the probe table below)")
for name, gap in CAPSTONE_GAPS.items():
    print(f"capstone gap, {name:<13} = {gap:.4f} nats   gap / sigma_stat = {gap / sigma_stat:.2f}   gap / band = {gap / band:.2f}   gap / bias = {gap / bias:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
names = ["gap,\nequal weights", "gap,\nweighted", "device\nstatistical σ", "device\nsystematic bias"]
mags = [EQUAL_WEIGHT_GAP_NATS, WEIGHTED_GAP_NATS, sigma_stat, bias]
bar_colors = [COLORS["muted"], COLORS["highlight"], COLORS["quantum"], COLORS["negative"]]
ax.bar(names, mags, color=bar_colors, alpha=0.9)
ax.axhline(band, color=COLORS["text"], lw=1.4, ls="--",
           label=f"band of a difference, √2·σ = {band:.4f} nats")
ax.set_yscale("log")
ax.set_ylim(top=bias * 4)
ax.set_ylabel("nats (log scale)")
ax.set_title("The gaps a device must resolve, beside the errors it has", pad=12)
for i, m in enumerate(mags):
    ax.text(i, m * 1.15, f"{m:.4f}", ha="center", zorder=5,
            bbox={"facecolor": COLORS["background"], "edgecolor": "none", "pad": 1.5})
ax.legend(loc="upper left", frameon=False)
plt.show()

**Read the figure.** Four bars and one dashed line, on a log scale, where equal ratios are equal distances. The blue bar is the error bar of one estimate, $\sigma_{\text{stat}} = 0.0087$ nat. The dashed line sits above it by the factor $\sqrt 2$: it is the band of a *difference* of two estimates, about $0.012$ nat, and it is the line to read the gaps against. The grey bar, the equal-weight gap of `04/15`, stands at about a third of that band ($0.36$ in the printout): one run of the device cannot tell the two ensembles apart with it. The rose bar, the weighted gap of `04/17`, passes the line by a little ($1.18$ bands). It is a larger gap than the equal-weight one, and it is not yet a detection, for which the usual rule of thumb asks two or three bands. The coral bar is the systematic bias, $0.39$ nat. The printout gives each gap as a fraction of it, $0.011$ and $0.037$: the bias is about ninety times the first gap and twenty-seven times the second.

Two cautions go with these numbers. They are the device's errors **for this probe**, a Bell pair, and not a property of the chip alone: the next cell changes the probe, and both the bias and the scatter move with it. And they are the errors of the offline FakeManila model; flip `USE_HARDWARE=True` and the bars report the literal chip's resolution.

The bias needs one more caveat. Each gap is a *difference* of two mutual informations, and a systematic offset that hits both ensembles alike cancels in that difference. It cancels only in part, as the next cell shows, because the offset grows with the mutual information itself. So the bias is the wall for reading an *absolute* QMI, while the limit for telling the two ensembles apart is the statistical band of the difference.

### Does the probe matter?

A fair question about the figure above: would a different prepared state give the same two error bars? We can measure it. The probe $R_y(\theta)$ followed by a CNOT prepares $\cos(\theta/2)\,|00\rangle + \sin(\theta/2)\,|11\rangle$. Its ideal mutual information runs from $0$ at $\theta = 0$, a product state, to $2\ln 2 \approx 1.386$ nats at $\theta = \pi/2$, the Bell pair. We repeat the tomography and the bootstrap for five values of $\theta$. This cell runs on the offline device model only, so it spends no time on a real processor.

In [ ]:
def probe_resolution(theta, shots=8192, n_boot=200, seed=0):
    """Ideal QMI, device QMI and bootstrap scatter (all in nats) for the probe Ry(theta); CX."""
    probe = QuantumCircuit(2)
    probe.ry(theta, 0)
    probe.cx(0, 1)
    probe_circuits = two_qubit_tomography_circuits(probe)
    names = list(probe_circuits)
    result = SamplerV2(mode=backend).run([pm.run(probe_circuits[s]) for s in names], shots=shots).result()
    counts = {s: result[i].data.c.get_counts() for i, s in enumerate(names)}
    p = np.cos(theta / 2) ** 2                      # weight of |00>; ideal QMI = 2 * H(p)
    ideal = 0.0 if p in (0.0, 1.0) else float(-2 * (p * np.log(p) + (1 - p) * np.log(1 - p)))
    device = quantum_mutual_information(density_from_counts(counts, n_qubits=2), dims=[2, 2], base=np.e)
    rng_boot = np.random.default_rng(seed)
    estimates = []
    for _ in range(n_boot):
        resampled = {}
        for s, c in counts.items():
            ks = list(c)
            ps = np.array([c[k] for k in ks], dtype=float)
            resampled[s] = {k: int(v) for k, v in zip(ks, rng_boot.multinomial(int(ps.sum()), ps / ps.sum()))}
        estimates.append(quantum_mutual_information(density_from_counts(resampled, n_qubits=2), dims=[2, 2], base=np.e))
    return ideal, float(device), float(np.std(estimates))

if is_real:
    print("Skipped on real hardware: this sweep would cost five more tomography jobs. Run it offline.")
else:
    rows = [probe_resolution(theta) for theta in np.linspace(0.0, np.pi / 2, 5)]
    print(" I ideal   I device    bias    sigma_stat   (nats)")
    for ideal, device, scatter in rows:
        print(f"  {ideal:.4f}    {device:.4f}   {ideal - device:+.4f}    {scatter:.4f}")
    table = np.array(rows)
    local_slopes = np.diff(table[:, 1]) / np.diff(table[:, 0])
    print("device QMI gained per nat of ideal QMI, between consecutive rows: "
          + ", ".join(f"{s:.2f}" for s in local_slopes))
    for name, gap in CAPSTONE_GAPS.items():
        print(f"=> a true gap of {gap:.4f} nats ({name}) would read as "
              f"{gap * local_slopes.min():.4f} to {gap * local_slopes.max():.4f} nats")

**Read the table.** Two columns move with the probe. The bias grows with the mutual information of the prepared state, from about zero for the product state (where finite counts and device noise leave the estimate a little above its ideal value of zero) to $0.39$ nats for the Bell pair: the Bell pair is the worst case for the bias, not the typical one. The statistical scatter changes as well, between $0.004$ and $0.010$ nats. So the bars of the figure describe *this probe on this device model*, and a statement about another state needs its own measurement. The capstone's gaps live in a pair of qutrits, which a two-qubit probe does not reproduce, so the comparison made in this notebook is one of orders of magnitude. Its states are also weakly correlated, with mutual informations below $0.08$ nat (`04/17`), under the second row of the table. The closer guide is the first row and the slope between the first two rows; closer to zero the slope was not measured.

The last two lines carry the consequence for a *difference*. If the bias were the same for every state, it would cancel exactly when we subtract two mutual informations. It is not the same: for each extra nat of ideal mutual information the device returns between $0.60$ nat, among the weakly correlated probes, and $0.81$ nat, near the Bell pair. So it shrinks a difference instead of leaving it intact: a true gap of $0.0044$ nat would read as $0.0026$ to $0.0035$, and one of $0.0145$ nat as $0.0087$ to $0.0117$. The bias neither hides a gap nor cancels: it scales it down by a fifth to two fifths, and by more where the correlation is weak. (These are the values of the offline FakeManila model.)

### A real run on IBM hardware

On **`ibm_marrakesh`** (2026-05-30), the device's I(A:B) for a Bell pair was **1.210 ± 0.012 (stat) nats**, with a systematic bias of **0.176 nats** below the ideal 1.386. The band of a difference is then $\sqrt 2 \times 0.012 \approx 0.017$ nat. The equal-weight gap ($0.0044$ nat) is about a quarter of that band and the weighted gap ($0.0145$ nat) about $0.85$ of it: these are the two ratios `04/17` printed under its first figure. Compared with the offline model, the real device has less bias ($0.39 \to 0.18$ nat) and more scatter ($0.0087 \to 0.012$ nat), and the reading is the same: the equal-weight gap is out of reach of one run, and the weighted gap sits at about one band. As above, these are the errors of a two-qubit Bell-pair probe. Set `USE_HARDWARE = True` to reproduce.

## 3 · What would it take?

So one gap is below the band and the other sits at it. The natural next question is quantitative: *how much* would we have to improve to resolve them? Take the statistical part first. The gap is a difference of two estimates, one per ensemble. If the two are independent and equally noisy, its statistical band is `√2·σ_stat`. The bootstrap scatter counts shot noise only, so this is the smallest the real band can be. To resolve a gap at about 3σ we need that band to be no larger than a third of the gap, `√2·σ_stat ≤ gap / 3`. Tomographic statistical error scales as `σ_stat ∝ 1/√(shots)`, so to shrink the band by a factor we must increase the shots by that factor *squared*. The required shot count therefore scales as `(√2·σ_stat / target)²`.

But — and this is the load-bearing caveat — **more shots only shrink the statistical band**. The *systematic* bias is a different animal: it comes from decoherence and read-out error, and it does not budge no matter how many times you sample. Driving it down needs error mitigation, or materially lower gate and read-out error rates — better hardware, not more of the same.

In [ ]:
shots_used = 8192
sigma_diff = np.sqrt(2) * sigma_stat         # a gap is a DIFFERENCE of two QMI estimates
print(f"band of the difference now    : {sigma_diff:.5f} nats  (sqrt(2) * sigma_stat)")
print(f"shots/setting now             : {shots_used:,}")
for name, gap in CAPSTONE_GAPS.items():
    target = gap / 3.0                       # aim: the band no larger than a third of the gap
    shots_needed = shots_used * (sigma_diff / target) ** 2
    print(f"gap of {gap:.4f} nats ({name}): target band {target:.5f} nats -> "
          f"{shots_needed:,.0f} shots/setting, for each of the two ensembles")
if not is_real:  # the slopes come from the offline probe sweep above
    print(f"once the device has rescaled the gap (slope {local_slopes[0]:.2f} between the two weakest probes): "
          f"about {1 / local_slopes[0] ** 2:.1f} times more shots")
print(f"bias of the Bell probe        : {bias:.4f} nats  (unaffected by shots)")
print("=> shots shrink the statistical band only. The bias limits an ABSOLUTE reading; in a")
print("   DIFFERENCE it does not cancel exactly, it rescales the gap (see the probe table).")

**Read the numbers.** For the equal-weight gap the statistical part calls for about $5.8 \times 10^5$ shots *per measurement setting, for each of the two ensembles*: seventy times the 8,192 of one run, a large but conceivable budget. For the weighted gap it calls for about $5.3 \times 10^4$, six to seven times one run. Turning the dial in `04/17` divided the bill by about eleven, the square of the ratio of the two gaps. Both counts are twice what a single estimate would need, because the band of a difference is $\sqrt{2}$ times wider, and with nine settings each is to be multiplied by nine for one ensemble.

Three things keep these counts from being a quote. They assume that the device returns the gap at full size, and the probe table said it does not. The printed line that follows the two counts corrects for it with the slope measured between the two weakest probes, $0.60$, and each count grows by a factor of about $2.8$; the capstone's states are weaker still, where the slope was not measured. They use the scatter of the Bell pair, $0.0087$ nat, while the weaker probes of the table scattered less, between $0.004$ and $0.008$ nat. And they are counts for a two-qubit probe, not for the pair of qutrits of the capstone. Read them as orders of magnitude.

More shots do nothing to the systematic bias, and what that bias costs depends on the question. For an *absolute* reading of the mutual information it is the wall: on the Bell probe it is about ninety times the equal-weight gap. Error mitigation or lower error rates might reduce it, at a cost in shots that this notebook did not measure. For the *difference* the capstone needs, the bias plays a smaller part: it scales the gap down (the probe table above) and leaves the statistical band as the binding constraint.

There is a loose kinship with `04/19` (De Palma, Marvian, Rouzé, Stilck França): there, noise limits what a shallow circuit can compute; here, it limits what the device can read. The two parts of the read-out error differ in one respect that matters. The shot noise can be paid down with more shots, and the bias cannot.

## 4 · The bottom line

Two statements close the course, one from the theory and one from the machine. Each comes with its numbers, and neither should be rounded into the other.

**What the theory established (Module 04).** A quantum coupling measure reads past the phase-locking value when its embedding carries more than the first harmonic. It reads as far as a classical statistic of the same harmonics and no further, and less far on the relative phase of the two harmonics. What quantum optimal transport brings is one common geometry: one state, one distance from independence, and the choice of a statistic turned into the choice of an embedding. That is a unification. On the systems of the capstone it is not a gain in sensitivity, and `04/18` argues that no deterministic embedding of classical data would change that.

**What the machine charges (this module).** The gaps that carried the capstone's comparison are small beside what a processor resolves today. With 8,192 shots per setting, the band of a difference is about 0.012 nat on the offline model and 0.017 nat on the real run. The equal-weight gap, 0.0044 nat, is a quarter to a third of it. The weighted gap, 0.0145 nat, is about one band: larger than the first, and not a detection. Resolving them at three bands would take about $5.8 \times 10^5$ and $5.3 \times 10^4$ shots per setting, for each ensemble, and more once the device has shrunk the gap. The bias, 0.18 to 0.39 nat on a Bell pair, rules out reading an absolute mutual information off the device without correcting it. And these are the costs of the read-out alone, measured on a two-qubit probe: preparing the qutrit states of the capstone on a device is the expedition of the exercise below.

**Read the two together.** Nothing in this course shows a quantum processor doing better than a classical computer on these signals. The quantities were computed classically, a classical statistic reads the same structure, and the device adds noise to the reading. What you hold is a framework that was put to a fair test, an exact account of what it reads and what it does not, and the first prices on the road to hardware: the resolution, the bias and the shot budget of a two-qubit read-out.

Two places stay open. One is named in `04/18`: data that are quantum to begin with, or a genuine quantum measurement or channel inside the pipeline, where a quantum measure could read what no classical statistic can. The other is real recordings, which `04/20` leaves as a question for you. You now have the questions to ask of any claim that the frontier has been crossed. Knowing exactly what a tool reads, and what reading it costs, lets you plan for a frontier instead of guessing at it.

## Your turn

**Warm-up (offline, a few seconds).** Section 3 rests on the rule that the scatter falls as one over the square root of the number of shots. Check it: in the measurement cell of section 2, change `shots=8192` to `shots=32768` and run it again. The rule predicts a `σ_stat` half as large. On the offline model you should find about $0.004$ nat where you had $0.0087$, close to the prediction. Put the value back to 8,192 before you go on, since the numbers quoted in this notebook assume it.

**Expedition (optional, heavy — not part of the offline run).** This one is a real expedition, flagged so you know it is *not* part of the lightweight offline run above. If you want to push past the Bell-pair probe and attempt the capstone itself on a device:

- Encode the richer multi-frequency qutrit of `04/15` into **2 qubits per oscillator** (four levels hold the three of a qutrit, with one left unused), giving a 4-qubit joint state for the pair.
- Attempt the two-ensemble separation on the **simulator first** — confirm you recover the two gaps, 0.0044 and 0.0145 nat, with exact (or high-shot) statistics before spending anything on hardware.
- Then use the **σ_stat you measured here** as a floor for the shots the on-hardware version would take, and list what that estimate leaves out: the larger tomography, the preparation of the states, and any error mitigation.

Be warned: this is a large job — four qubits, full tomography, many settings, very high shot counts — well beyond a casual free-tier run. Treat it as a design exercise in *what a real demonstration would cost*, which is itself the lesson.

## What you built

- An **instrument reading**: `I(A:B)` of a Bell pair on a device, with a bootstrap error bar that costs no extra job, and its bias against the ideal value.
- A **probe table** showing that both errors depend on the prepared state, and that the device rescales a difference where one might have hoped the bias would cancel.
- A **budget**: the shots a device would need to resolve each gap of the capstone, and what shots cannot buy.
- The **bottom line of the course**, in two statements you can defend number by number.

## The journey, end to end

Look back at the whole arc. You began in `00` with a pile of mass to move and one line of code. In `01` you built the language: qubits, density matrices, composite systems, and entanglement. In `02` you turned entropy into geometry: Fisher–Rao, von Neumann entropy, mutual information, the Bures distance. In `03` you learned classical optimal transport — couplings, the Wasserstein distance, Sinkhorn — and watched it shake hands with information geometry. In `04` you lifted all of it into **quantum** optimal transport, reconciled its three faces, and ran the experiments that measured how far a quantum coupling measure reads past PLV, with a classical statistic beside it at every step. And in `05` you met real hardware: tomography, fidelity, mutual information, the Bures distance — each measured on a device — culminating, here, in an honest map of the frontier.

That is the arc: from a pile of mass, through the geometry of information and the mathematics of transport, to a framework you tested fairly and a clear-eyed account of what it reads, what it does not, and what a machine charges for it. You did the hard version every step — you built the objects, validated them against closed forms, and refused to round a modest result up into a victory or down into a failure. Thank you for walking the whole road. The frontier is now yours to push.

## References

- G. De Palma, M. Marvian, C. Rouzé & D. Stilck França, "Limitations of variational quantum algorithms: a quantum optimal transport approach", *PRX Quantum* **4**, 010309 (2023). DOI:10.1103/PRXQuantum.4.010309 — arXiv:2204.03455. (Four authors — the frontier result of `19`.)
- J. A. Smolin, J. M. Gambetta & G. Smith (2012), "Efficient Method for Computing the Maximum-Likelihood Quantum State from Measurements with Additive Gaussian Noise," *Phys. Rev. Lett.* **108**, 070502, doi:10.1103/PhysRevLett.108.070502.

**Previous:** `notebooks/05_RealQuantumHardware/05_geometry_on_real_hardware_bures.ipynb`. *Course end.*